# 05 — Независимый replay, парный анализ validation и фиксация решений

Только Google Colab, CPU, runtime 2026.07 / Python 3.12. Откройте новый независимый runtime.
Сначала опубликуйте evaluator и укажите его полный SHA в GIT_REF. SHA обучения старых моделей остаётся прежним.
Реальное обучение отсутствует. Полный pytest включает маленькие синтетические обучения.
См. docs/POSTPILOT_PROTOCOL.md. Новый код ещё не подтверждён выполнением в Colab.

In [ ]:
import sys
import os
import re
import json
import subprocess
import tempfile
from pathlib import Path

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Запуск разрешён только в Google Colab, не в локальном runtime") from error
if not Path("/content").is_dir() or "COLAB_RELEASE_TAG" not in os.environ:
    raise RuntimeError("Нужна управляемая сессия Google Colab с каталогом /content")
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Нужен Python 3.11/3.12. Выберите Runtime > Change runtime type > Runtime Version 2026.07 (Python 3.12).")

GITHUB_REPOSITORY = "Go9shi/Science_Paper"
GIT_REF = "main"  # First publish the new evaluator; then pin its full SHA here.
MODEL_GIT_SHA = "1e6795abb7c1c59f002f39fd2ec30a063cf6592f"
USE_GITHUB_SECRET = False
CHECK_RL_COMPATIBILITY = True
RUN_TESTS = True  # Full tests include tiny SYNTHETIC training, never real PPO training.
SOURCE_RUN_ID = "20261107T154124Z-9f1f7c1e"
BATCH_ID = "20261115T100446Z-757eee8d"
NOTEBOOK_REVISION = "locked-evaluation-v1"
RUN_REPLAY = False
INDEPENDENT_SESSION = False  # Set True only in a new independent Colab runtime.
RUN_ANALYSIS = False
SEAL_DECISIONS = False
if any(type(v) is not bool for v in (RUN_TESTS, RUN_REPLAY, INDEPENDENT_SESSION, RUN_ANALYSIS, SEAL_DECISIONS)):
    raise ValueError("Switches must be Python booleans")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
REPO_URL = f"https://github.com/{GITHUB_REPOSITORY}.git"
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", GITHUB_REPOSITORY):
    raise ValueError("Ожидается repository в формате owner/name")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]*", GIT_REF):
    raise ValueError("Некорректный GIT_REF")
if not re.fullmatch(r"[A-Za-z0-9_-]+", SOURCE_RUN_ID):
    raise ValueError("Некорректный SOURCE_RUN_ID")

REPO_ROOT = Path(tempfile.mkdtemp(prefix="Science_Paper-stage5-", dir="/content"))
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
with tempfile.TemporaryDirectory(prefix="stage1-auth-", dir="/content") as auth_dir:
    if USE_GITHUB_SECRET:
        from google.colab import userdata
        try:
            git_env["STAGE1_GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
        except Exception as error:
            raise RuntimeError("Разрешите доступ к Colab Secret GITHUB_TOKEN с правом чтения репозитория") from error
        if not git_env["STAGE1_GITHUB_TOKEN"]:
            raise RuntimeError("GITHUB_TOKEN пуст")
        askpass = Path(auth_dir) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['STAGE1_GITHUB_TOKEN'])\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        git_env["GIT_ASKPASS"] = str(askpass)
    try:
        subprocess.run(["git", "-c", "credential.helper=", "clone", "--quiet", "--no-checkout",
                        REPO_URL, str(REPO_ROOT)], env=git_env, check=True, capture_output=True, text=True, timeout=180)
        if re.fullmatch(r"[0-9a-fA-F]{7,40}", GIT_REF):
            # Full clone already contains reachable history; resolve short SHAs locally.
            resolved = subprocess.run(
                ["git", "-C", str(REPO_ROOT), "rev-parse", "--verify", f"{GIT_REF}^{{commit}}"],
                capture_output=True, text=True,
            )
            if resolved.returncode == 0:
                SOURCE_COMMIT = resolved.stdout.strip()
            elif len(GIT_REF) < 40:
                raise ValueError("Short SHA was not found or is ambiguous. Use the full 40-character commit SHA.")
            else:
                subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                               env=git_env, check=True, capture_output=True, text=True, timeout=180)
                SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        else:
            subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                           env=git_env, check=True, capture_output=True, text=True, timeout=180)
            SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
        raise RuntimeError("Не удалось получить код GitHub. Проверьте сеть, GIT_REF и доступ; для private используйте Secret.") from error
    finally:
        git_env.pop("STAGE1_GITHUB_TOKEN", None)
        git_env.pop("GIT_ASKPASS", None)

PROJECT_ROOT = REPO_ROOT / "research"
required = ["src/postpilot.py", "src/paired_analysis.py", "src/out_of_sample.py", "tests/test_postpilot.py", "configs/evaluation_protocol.json", "docs/POSTPILOT_PROTOCOL.md", "notebooks/05_validation_and_freeze.ipynb", "AGENTS.md", "src/__init__.py", "src/data.py", "src/volatility.py", "src/validation.py", "src/environment.py",
            "tests/test_data.py", "tests/test_volatility.py", "tests/test_validation.py", "tests/test_environment.py",
            "requirements-colab.txt", "requirements-rl-check.txt", "docs/METHODOLOGY.md", "docs/STAGE1_VALIDATION.md", "docs/COLAB_ENVIRONMENT.md",
            "notebooks/01_data_preparation.ipynb"]
if not CHECK_RL_COMPATIBILITY:
    raise RuntimeError("Post-pilot evaluation requires CHECK_RL_COMPATIBILITY=True")
required += ["src/portfolio.py", "src/environment_data.py", "src/portfolio_env.py", "src/env_validation.py",
             "tests/test_portfolio_env.py", "docs/STAGE2_VALIDATION.md", "notebooks/02_environment_validation.ipynb"]
required += ["src/training_config.py", "src/training_data.py", "src/ppo_training.py", "src/financial_metrics.py",
             "src/regularization.py", "src/replay_audit.py", "configs/ppo_smoke.json", "configs/ppo_pilot.json",
             "tests/test_ppo_training.py", "docs/STAGE3_VALIDATION.md", "notebooks/03_ppo_training.ipynb"]
required += ["src/pilot.py", "src/pilot_plan.py", "tests/test_pilot.py", "configs/pilot_plan.json",
             "docs/PILOT_PROTOCOL.md", "docs/COLAB_QUICKSTART.md", "notebooks/04_pilot_experiments.ipynb"]
missing = [name for name in required if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Commit {SOURCE_COMMIT} не содержит актуальный этап 4: {missing}. "
                            "Опубликуйте файлы и выберите содержащий их ref; локальные файлы не подменяют GitHub.")
source_notebook = json.loads((PROJECT_ROOT / "notebooks/05_validation_and_freeze.ipynb").read_text(encoding="utf-8"))
if source_notebook["metadata"].get("postpilot_revision") != NOTEBOOK_REVISION:
    raise RuntimeError("Версия открытого notebook не соответствует checkout; откройте notebook из выбранного commit")
print("GitHub:", GITHUB_REPOSITORY)
print("Resolved commit:", SOURCE_COMMIT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## Google Drive и исходный Stage 1

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Drive не подключён: каталог MyDrive недоступен")
OUTPUT_ROOT = DRIVE_ROOT / "ppo_turnover_research" / "stage5"
PREFLIGHT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
PREFLIGHT_DIR = OUTPUT_ROOT / "preflight" / PREFLIGHT_ID
try:
    PREFLIGHT_DIR.mkdir(parents=True, exist_ok=False)
    probe = PREFLIGHT_DIR / "drive-probe.txt"
    probe_value = uuid4().hex
    probe.write_text(probe_value, encoding="utf-8")
    if probe.read_text(encoding="utf-8") != probe_value:
        raise OSError("Drive read-back mismatch")
    probe.unlink()
except OSError as error:
    raise RuntimeError(f"Нет надёжного чтения/записи Google Drive: {OUTPUT_ROOT}") from error
print("Drive read/write: OK")
print("Preflight logs:", PREFLIGHT_DIR)
STAGE1_RUN = DRIVE_ROOT / "ppo_turnover_research" / "stage1" / "runs" / SOURCE_RUN_ID
if not (STAGE1_RUN / "metadata.json").is_file():
    raise FileNotFoundError(f"Complete Stage 1 directory is unavailable: {STAGE1_RUN}")
print("Stage 1 source:", STAGE1_RUN)

## Зависимости без замены системного PyTorch

In [ ]:
import importlib.metadata

# Read distribution metadata only: do not import NumPy/Pandas before pip.
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

before_install = {name: installed_version(name) for name in (
    "numpy", "pandas", "torch", "torchvision", "torchaudio", "google-colab",
    "ipython", "jedi", "ibis-framework", "toolz", "numba", "exchange-calendars",
)}
(PREFLIGHT_DIR / "versions-before-install.json").write_text(
    json.dumps(before_install, indent=2), encoding="utf-8")
install_args = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--report", str(PREFLIGHT_DIR / "pip-install-report.json"),
                "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
if CHECK_RL_COMPATIBILITY:
    if before_install["torch"] is None:
        raise RuntimeError("В runtime нет PyTorch. Выберите Colab 2026.07; автоматически скачивать/заменять torch не будем.")
    # Keep the exact preinstalled build, including its CUDA/CPU local suffix.
    constraint = PREFLIGHT_DIR / "runtime-constraints.txt"
    constraint.write_text(f"torch=={before_install['torch']}\n", encoding="utf-8")
    install_args += ["-r", str(PROJECT_ROOT / "requirements-rl-check.txt"), "-c", str(constraint)]
install = subprocess.run(install_args, text=True, capture_output=True)
(PREFLIGHT_DIR / "pip-install.log").write_text(install.stdout + "\n" + install.stderr, encoding="utf-8")
if install.returncode != 0:
    print(install.stdout[-4000:])
    print(install.stderr[-4000:])
    raise RuntimeError(f"Установка не прошла: {PREFLIGHT_DIR / 'pip-install.log'}; не обходите конфликт resolver.")
if CHECK_RL_COMPATIBILITY and installed_version("torch") != before_install["torch"]:
    raise RuntimeError("Версия PyTorch изменилась; проверка окружения остановлена")
print("Установка завершена без --upgrade; лог:", PREFLIGHT_DIR / "pip-install.log")

In [ ]:
import importlib
import importlib.metadata
import textwrap

requirements = {}
requirement_files = [PROJECT_ROOT / "requirements-colab.txt"]
if CHECK_RL_COMPATIBILITY:
    requirement_files.append(PROJECT_ROOT / "requirements-rl-check.txt")
for req_file in requirement_files:
    for line in req_file.read_text(encoding="utf-8").splitlines():
        if "==" in line and not line.lstrip().startswith("#"):
            package, version = line.strip().split("==")
            requirements[package] = version
module_names = {"stable-baselines3": "stable_baselines3"}
for package, version in requirements.items():
    installed = importlib.metadata.version(package)
    loaded = getattr(sys.modules.get(module_names.get(package, package)), "__version__", installed)
    if installed != version or loaded != version:
        raise RuntimeError(f"{package}: installed={installed}, loaded={loaded}, expected={version}. "
                           "Перезапустите сеанс Colab и выполните notebook сверху.")

project_env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT), "MPLBACKEND": "Agg"}
smoke_source = textwrap.dedent("""
import importlib, json
names = __MODULES__ + ['src.data', 'src.volatility', 'src.validation', 'src.environment', 'src.portfolio', 'src.environment_data', 'src.portfolio_env', 'src.env_validation', 'src.training_config', 'src.training_data', 'src.regularization', 'src.financial_metrics', 'src.ppo_training', 'src.replay_audit', 'src.pilot_plan', 'src.pilot', 'src.paired_analysis', 'src.postpilot', 'src.out_of_sample']
modules = {name: importlib.import_module(name) for name in names}
from src.environment import dependency_closure
result = {
    'module_paths': {n: str(m.__file__) for n, m in modules.items()},
    'dependency_closure': dependency_closure(__ROOTS__),
    'future_stack': {'status': 'not_requested'},
}
if __CHECK_RL__:
    import numpy as np
    import torch
    from packaging.version import Version
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_checker import check_env
    if not (Version('2.3') <= Version(torch.__version__) < Version('3.0')):
        raise RuntimeError('PyTorch does not satisfy SB3 2.7.0')
    if Version(torch.__version__).base_version != '2.11.0':
        raise RuntimeError('Expected preinstalled PyTorch 2.11.0 for the selected 2026.07 baseline')
    array = np.array([1.0, 2.0], dtype=np.float32)
    recovered = torch.from_numpy(array).add(1).numpy()
    if not np.array_equal(recovered, np.array([2.0, 3.0], dtype=np.float32)):
        raise RuntimeError('NumPy/PyTorch CPU interoperability check failed')
    result['future_stack'] = {
        'status': 'passed', 'torch': torch.__version__, 'cuda_build': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(), 'numpy_torch_bridge': 'passed',
        'gymnasium': modules['gymnasium'].__version__,
        'stable_baselines3': modules['stable_baselines3'].__version__,
        'PPO_import': 'passed', 'model_created': False, 'training_performed': False,
        'project_environment_created': False,
    }
print(json.dumps(result, indent=2))
""")
smoke_source = (smoke_source.replace("__MODULES__", repr([module_names.get(p, p) for p in requirements]))
                .replace("__ROOTS__", repr(list(requirements) + (["torch"] if CHECK_RL_COMPATIBILITY else [])))
                .replace("__CHECK_RL__", repr(CHECK_RL_COMPATIBILITY)))
smoke = subprocess.run([sys.executable, "-c", smoke_source], cwd=PROJECT_ROOT,
                       env=project_env, capture_output=True, text=True)
(PREFLIGHT_DIR / "imports.log").write_text(smoke.stdout + "\n" + smoke.stderr, encoding="utf-8")
if smoke.returncode != 0:
    print(smoke.stderr[-4000:])
    raise RuntimeError(f"Ошибка import/ABI preflight; см. {PREFLIGHT_DIR / 'imports.log'}")
smoke_report = json.loads(smoke.stdout)

# Equivalent CLI command: python -m pip check (in this Colab kernel's Python).
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
(PREFLIGHT_DIR / "pip-check.txt").write_text(pip_check.stdout + "\n" + pip_check.stderr, encoding="utf-8")
print(pip_check.stdout or pip_check.stderr)
policy_source = (
    "import json, sys; from src.environment import classify_pip_check; "
    "payload=json.load(sys.stdin); print(json.dumps(classify_pip_check(payload['output'], payload['code'])))"
)
policy = subprocess.run([sys.executable, "-c", policy_source], input=json.dumps({
    "output": pip_check.stdout + "\n" + pip_check.stderr, "code": pip_check.returncode,
}), cwd=PROJECT_ROOT, env=project_env, capture_output=True, text=True, check=True)
classification = json.loads(policy.stdout)
environment = {
    "schema": 2, "profile": "env-v2-colab-2026.07", "rl_check_requested": CHECK_RL_COMPATIBILITY,
    "direct_versions": {p: importlib.metadata.version(p) for p in requirements},
    "versions_before_install": before_install,
    "project_imports": "passed", "subprocess_module_paths": smoke_report["module_paths"],
    "dependency_closure": smoke_report["dependency_closure"],
    "future_stack": smoke_report["future_stack"], "pip_check_policy": classification,
    "pip_check_returncode": pip_check.returncode, "drive_read_write": "passed",
    "python": sys.version, "colab_release_tag": os.environ["COLAB_RELEASE_TAG"],
}
(PREFLIGHT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Фактические версии:", json.dumps(environment["direct_versions"], indent=2))
print("Dependency closure:", smoke_report["dependency_closure"]["status"])
print("Global pip check:", classification["status"])
print("Future stack:", json.dumps(environment["future_stack"], indent=2))
if smoke_report["dependency_closure"]["status"] != "passed" or classification["status"] == "blocked":
    raise RuntimeError(f"Неустранённый или новый конфликт; изучите {PREFLIGHT_DIR / 'environment.json'}")
print("Project dependency preflight passed; известные предупреждения, если есть, сохранены.")

## Новая попытка и provenance

In [ ]:
import shutil
import importlib
import pandas as pd
from IPython.display import display
os.chdir(PROJECT_ROOT)
for name, module in list(sys.modules.items()):
    if name == "src" or name.startswith("src."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(PROJECT_ROOT.resolve()):
            raise RuntimeError("src belongs to an older checkout; delete runtime and start from the top")
sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
from src.data import sha256_file, write_json
from src.validation import pytest_summary, verify_file_map
from src.postpilot import read_json, seal_report
PROTOCOL = read_json(PROJECT_ROOT / "configs/evaluation_protocol.json")
if PROTOCOL["training_sha"] != MODEL_GIT_SHA or PROTOCOL["batch_id"] != BATCH_ID:
    raise ValueError("Model SHA / BATCH_ID differs from the evaluation protocol")
BATCH_DIR = DRIVE_ROOT / "ppo_turnover_research/stage4/batches" / BATCH_ID
if not (BATCH_DIR / "batch_report.json").is_file():
    raise FileNotFoundError(f"Source batch unavailable: {BATCH_DIR}")
dirty = subprocess.check_output(["git", "-C", str(REPO_ROOT), "status", "--porcelain", "--", "research"], text=True)
if dirty.strip():
    raise RuntimeError("Evaluator checkout has uncommitted changes; publish and pin the exact code")
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
SESSION_DIR = OUTPUT_ROOT / "sessions" / SESSION_ID
SESSION_DIR.mkdir(parents=True, exist_ok=False)
source_paths = sorted(set([
    *PROJECT_ROOT.glob("src/*.py"), *PROJECT_ROOT.glob("tests/test_*.py"),
    *PROJECT_ROOT.glob(".agents/skills/*/SKILL.md"), *PROJECT_ROOT.glob("configs/*.json"),
    *PROJECT_ROOT.glob("docs/*.md"), *PROJECT_ROOT.glob("notebooks/*.ipynb"),
    PROJECT_ROOT / "AGENTS.md", PROJECT_ROOT / "requirements-colab.txt", PROJECT_ROOT / "requirements-rl-check.txt",
]))
for path in source_paths:
    target = SESSION_DIR / "provenance" / path.relative_to(PROJECT_ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, target)
subprocess.run(["git", "-C", str(REPO_ROOT), "bundle", "create", str(SESSION_DIR / "source.bundle"), "HEAD"], check=True)
for path in PREFLIGHT_DIR.iterdir():
    if path.is_file():
        shutil.copy2(path, SESSION_DIR / path.name)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
(SESSION_DIR / "pip-freeze.txt").write_text(freeze.stdout, encoding="utf-8")
identity = {"independence": "user attestation; original training boot identity was not recorded",
    "kernel_boot_id": Path("/proc/sys/kernel/random/boot_id").read_text().strip(), "process_id": os.getpid()}
write_json(SESSION_DIR / "session_identity.json", identity)
session = {"status": "prepared", "evaluator_sha": SOURCE_COMMIT, "training_sha": MODEL_GIT_SHA,
    "session_id": SESSION_ID, "source_batch": str(BATCH_DIR), "training_performed": False,
    "environment": environment, "tests": {"status": "not_run"}, "test_opened": False}
write_json(SESSION_DIR / "session.json", session)
print("Evaluator SHA:", SOURCE_COMMIT)
print("Original model-training SHA:", MODEL_GIT_SHA)
print("SESSION_ID:", SESSION_ID)
print("SESSION_DIR:", SESSION_DIR)

## Полный pytest до реальных данных

In [ ]:
if RUN_TESTS:
    if (SESSION_DIR / "test_gate.json").exists():
        raise RuntimeError("This attempt already has immutable test evidence; rerun section 4 to create a new attempt")
    result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q",
                             f"--junitxml={SESSION_DIR / 'pytest.xml'}"], cwd=PROJECT_ROOT,
                            env={**project_env, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}, capture_output=True, text=True)
    (SESSION_DIR / "pytest.log").write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
    print(result.stdout)
    try:
        if result.returncode:
            raise RuntimeError("Full pytest failed; inspect pytest.log, do not proceed")
        summary = pytest_summary(SESSION_DIR / "pytest.xml")
        import xml.etree.ElementTree as ET
        cases = {c.get("name") for c in ET.parse(SESSION_DIR / "pytest.xml").getroot().iter("testcase")}
        required_cases = {"test_pilot_plan_equal_budgets", "test_pilot_disabled_does_not_train",
                          "test_pilot_resume_preserves_attempts_and_skips_completed",
                          "test_gymnasium_and_sb3_checkers", "test_tiny_synthetic_training_saves_and_reloads",
                          "test_paired_identical_paths_have_exact_zero_interval",
                          "test_test_access_disabled_before_reading_files",
                          "test_nonexact_replay_cannot_seal_decisions",
                          "test_test_evaluation_is_explicit_and_preserves_checkpoint_config"}
        if not required_cases.issubset(cases):
            raise RuntimeError("Required Stage 4/PPO/checker cases were not collected")
        session["tests"] = {**summary, "exit_code": 0}
        write_json(SESSION_DIR / "test_gate.json", {"git_sha": SOURCE_COMMIT,
            "tests": session["tests"], "pytest_xml_sha256": sha256_file(SESSION_DIR / "pytest.xml")})
    except Exception as error:
        session.update(status="tests_failed", error=str(error),
                            tests={"status": "failed", "exit_code": result.returncode})
        write_json(SESSION_DIR / "attempt.json", session)
        raise
else:
    session["tests"] = {"status": "not_run_preview", "reason": "RUN_TESTS=False; real data/training blocked"}
write_json(SESSION_DIR / "attempt.json", session)
print("Actual tests:", session["tests"])
if not RUN_TESTS:
    raise RuntimeError("Full tests are required before replay, analysis, sealing, or test access")

## Независимый replay всех 27 моделей
В первой ячейке включите RUN_REPLAY=True и INDEPENDENT_SESSION=True только после создания новой сессии.
Сравниваются SHA256, исполнение исходных действий, новые действия на исходных состояниях,
полная новая траектория/метрики и две загрузки внутри нового runtime. Допуски не используются.
diagnostic_complete при несовпадении НЕ означает passed и блокирует фиксацию/test.

In [ ]:
from src.training_data import load_development_market
from src.postpilot import audit_pilot_validation
if RUN_REPLAY:
    validation = load_development_market(STAGE1_RUN, "validation")
    try:
        replay = audit_pilot_validation(BATCH_DIR, validation, PROTOCOL, SESSION_DIR / "replay",
            evaluator_sha=SOURCE_COMMIT, independent_session=INDEPENDENT_SESSION)
        session["reproducibility_status"] = replay["status"]
    except BaseException as error:
        session.update(status="replay_failed", error=str(error))
        seal_report(SESSION_DIR, "session.json", session)
        raise
else:
    session["reproducibility_status"] = "not_run"
    print("Replay disabled; no exact-replay pass claimed")
write_json(SESSION_DIR / "session.json", session)

## Парный анализ сохранённого validation
RUN_ANALYSIS=True использует исходные проверенные финансовые CSV, не заменяет их новыми replay-результатами.
Высокая sigma > train sigma_ref, низкая <=. Bootstrap: общие индексы для всех режимов/seeds,
2000 повторов, длина блока 20 с sensitivity 10/40. Это разведочная оценка после выбора на validation.
Дни разных режимов не склеиваются для CAGR/просадки. Seed sample std — не доверительный интервал.

In [ ]:
from src.postpilot import analyze_validation
if RUN_ANALYSIS:
    try:
        analysis = analyze_validation(BATCH_DIR, PROTOCOL, SESSION_DIR / "analysis")
        session["analysis_status"] = analysis["status"]
    except BaseException as error:
        session.update(status="analysis_failed", error=str(error))
        seal_report(SESSION_DIR, "session.json", session)
        raise
    for name in ("seed_stability", "regime_metrics", "turnover_mechanism_by_seed", "bootstrap_intervals"):
        display(pd.read_csv(SESSION_DIR / "analysis" / (name + ".csv")))
else:
    session["analysis_status"] = "not_run"
write_json(SESSION_DIR / "session.json", session)

## Неизменяемые решения до test
SEAL_DECISIONS=True создаёт lock только после точного независимого replay всех 27 моделей и анализа.
Зафиксированы 9 final моделей (3 режима × 3 seed), исходные данные, метрики, комиссии,
порог волатильности, bootstrap, evaluator SHA. Нового обучения/выбора по test нет.
Скопируйте напечатанный SHA256 lock во внешний журнал/репозиторий ДО открытия notebook 06.
Не подбирайте допуски или гиперпараметры при неуспешном replay.

In [ ]:
from src.postpilot import freeze_decisions
if SEAL_DECISIONS:
    if session.get("reproducibility_status") != "exact_replay_passed" or session.get("analysis_status") != "analysis_complete":
        session["status"] = "test_blocked_incomplete_or_nonexact_validation"
        print("TEST BLOCKED: exact independent replay and paired analysis must complete first")
    else:
        freeze_decisions(BATCH_DIR, STAGE1_RUN, SESSION_DIR, PROTOCOL,
                         evaluator_sha=SOURCE_COMMIT, project_root=PROJECT_ROOT)
        session["status"] = "decisions_sealed_test_unopened"
        print("DECISION_SESSION_ID =", repr(SESSION_ID))
        print("LOCK_SHA256 =", repr(sha256_file(SESSION_DIR / "decision_lock.json")))
else:
    session["status"] = "validation_workflow_complete_without_seal"
    print("No decision lock created; test remains blocked")
seal_report(SESSION_DIR, "session.json", session)
print("Actual status:", session["status"])
print("Artifacts:", SESSION_DIR)
print("No test observations opened. No real PPO training performed.")